# 第 06 章　决策树与随机森林

> **本章问题**：一棵树为什么容易把训练区“记住”，随机森林又怎样缓解这个问题？

线性回归只能给变量配一条平均斜率，但地表变量之间常常不是这么简单。同样的气温，在不同地形或植被条件下，可能对应不同的 LST。决策树可以按阈值一层层切分样本，随机森林再把许多棵树的结果平均起来。

我们先观察树长得多深会发生什么，再在固定空间折和测试区上比较单树与森林。


## 6.1　先把数据和划分核对一遍

我们继续沿用第 04 章的固定空间划分。训练区内部是 5 个空间折，中南部连续区域是独立测试区，只有定稿后才看测试成绩。

这一格先不训练模型，只核对训练/测试像元数、每折大小，以及两区的目标均值差异。


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_models as gm
import geoml_style as gs

gs.setup()
T0 = time.perf_counter()
print("Python:", sys.executable)
print("Conda:", os.environ.get("CONDA_PREFIX"))
print("项目根目录:", PROJECT)

In [ ]:
df = gd.load_splits()
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print(gd.split_summary(df).to_string(index=False))
print()
print("训练区目标均值: {:.2f}，标准差: {:.2f}".format(train_df["lst"].mean(), train_df["lst"].std()))
print("测试区目标均值: {:.2f}，标准差: {:.2f}".format(test_df["lst"].mean(), test_df["lst"].std()))
print("训练区缺测: ", train_df[gm.FEATURES].isna().sum().to_dict())

## 6.2　一棵回归树怎样做决定？

回归树不拟合一条斜率，而是在特征上找一个阈值，把样本切成左右两组。它会挑那个能最大幅度降低组内平方误差的切分，然后继续往下分。

下面这个公式写的就是“切完以后两组的误差加起来有多少”。树会寻找让这个值下降最多的特征和阈值。


In [ ]:
shallow = gm.tree_model(max_depth=3)
shallow.fit(train_df[gm.FEATURES], train_df["lst"])
shallow_pred_train = shallow.predict(train_df[gm.FEATURES])
shallow_train = gm.metrics(train_df["lst"], shallow_pred_train)

tree = shallow.named_steps["model"]
root_feature = gm.FEATURES_CN[tree.tree_.feature[0]]
print("深度 3：叶子数 = {}，训练区 RMSE = {:.3f}，MAE = {:.3f}，R2 = {:.3f}".format(
    tree.get_n_leaves(), shallow_train["RMSE"], shallow_train["MAE"], shallow_train["R2"]))
print("根节点分裂特征 = {}，阈值 = {:.3f}".format(root_feature, tree.tree_.threshold[0]))

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 9.2))
annotations = plot_tree(
    tree,
    feature_names=gm.FEATURES_CN,
    filled=True,
    rounded=True,
    impurity=True,
    precision=2,
    fontsize=10,
    node_ids=False,
    ax=ax,
)
# 把 sklearn 的英文节点文字换成中文，保持每个节点的信息量不变
for ann in annotations:
    txt = (ann.get_text()
           .replace("squared_error", "方差")
           .replace("samples", "样本数")
           .replace("value", "均值"))
    ann.set_text(txt)
for label in ax.texts:
    branch = label.get_text().strip()
    if branch == "True":
        label.set_text("是")
    elif branch == "False":
        label.set_text("否")
ax.set_title("甘南县训练区：限制为 3 层的回归树（左支为“是”、右支为“否”；节点内为 方差 / 样本数 / 平均 LST）", fontsize=15, pad=18)
gs.save(fig, "06_shallow_tree.png")
plt.show()

**读图**：根节点只用了一个条件——气温 ≤ 4.0（阈值 3.999），把 5796 个训练像元分成 3668 个（左支“是”，平均 LST 11.11）和 2128 个（右支“否”，平均 LST 13.40）。第二层继续按气温、降水、高程、土壤水分切分，8 个叶子的平均 LST 从 10.68 到 15.08，暖色叶子集中在“高温、低土壤水分”一侧。深度 3 的训练区 RMSE 为 1.152、R² 为 0.546，看起来像不错的模型——但这是**训练样本**上的拟合。节点里的“方差”逐层变小，正是分裂准则在起作用；它保证的是训练区内部更纯，不保证换一片区域还准。

## 6.3　让树长深一点，看看会发生什么

我们把 `max_depth` 从 1 逐步增加到 12。每个深度都记录两件事：固定空间折上的验证误差，以及整块训练区上的训练误差。

选深度只看训练区里的空间折，独立测试区先锁好。`min_samples_leaf=5` 保持不变，这一轮我们只研究树深的影响。


In [ ]:
depths = list(range(1, 13))
rows = []
for d in depths:
    cv = gm.cv_scores(lambda d=d: gm.tree_model(max_depth=d), train_df)
    fitted = gm.tree_model(max_depth=d).fit(train_df[gm.FEATURES], train_df["lst"])
    train_rmse = gm.metrics(train_df["lst"], fitted.predict(train_df[gm.FEATURES]))["RMSE"]
    rows.append({
        "深度": d,
        "训练RMSE": train_rmse,
        "CV_RMSE_均值": cv["RMSE"].mean(),
        "CV_RMSE_标准差": cv["RMSE"].std(ddof=1),
        "CV_MAE_均值": cv["MAE"].mean(),
        "CV_MAE_标准差": cv["MAE"].std(ddof=1),
        "CV_R2_均值": cv["R2"].mean(),
        "CV_R2_标准差": cv["R2"].std(ddof=1),
    })

depth_curve = pd.DataFrame(rows)
depth_curve.to_csv(gd.TABLES / "06_depth_curve.csv", index=False, encoding="utf-8-sig")
best_depth = int(depth_curve.loc[depth_curve["CV_RMSE_均值"].idxmin(), "深度"])
print(depth_curve.round(3).to_string(index=False))
print()
print("按训练区空间 CV 的 RMSE 折均值，最佳深度 =", best_depth)

In [ ]:
best_row = depth_curve[depth_curve["深度"] == best_depth].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.6), gridspec_kw={"width_ratios": [1.55, 1.0]})

ax = axes[0]
ax.plot(depths, depth_curve["训练RMSE"], "--o", color="#6b7280", ms=4.5,
        label="全训练区拟合：训练误差")
ax.plot(depths, depth_curve["CV_RMSE_均值"], "-o", color="#2a6f97", ms=5,
        label="5 个空间折：验证误差")
ax.fill_between(depths,
                depth_curve["CV_RMSE_均值"] - depth_curve["CV_RMSE_标准差"],
                depth_curve["CV_RMSE_均值"] + depth_curve["CV_RMSE_标准差"],
                color="#2a6f97", alpha=0.16, linewidth=0, label="CV 折间 ±1 标准差")
ax.axvline(best_depth, color="#b23a48", ls=":", lw=1.4)
ax.scatter([best_depth], [best_row["CV_RMSE_均值"]], marker="*", s=190,
           color="#b23a48", edgecolor="white", linewidth=0.7, zorder=6)
ax.annotate("选中深度 {}\nCV RMSE = {:.2f}".format(best_depth, best_row["CV_RMSE_均值"]),
            xy=(best_depth, best_row["CV_RMSE_均值"]), xytext=(14, 34),
            textcoords="offset points", fontsize=9.2, color="#b23a48",
            arrowprops=dict(arrowstyle="->", color="#b23a48", lw=1.0))
ax.set_xlabel("树的最大深度 max_depth")
ax.set_ylabel("RMSE（原始像元值）")
ax.set_xticks(depths)
ax.set_title("(a) 训练误差与空间验证误差随树深的变化")
ax.grid(axis="y", color=gs.GRID, lw=0.6, alpha=0.85)
ax.set_axisbelow(True)
ax.legend(loc="upper right", fontsize=8.6)

ax2 = axes[1]
ax2.errorbar(depths, depth_curve["CV_R2_均值"], yerr=depth_curve["CV_R2_标准差"],
             fmt="-o", color="#3a7d44", ecolor="#3a7d44", elinewidth=0.9,
             capsize=2.5, ms=4.5, alpha=0.9)
ax2.axvline(best_depth, color="#b23a48", ls=":", lw=1.4)
ax2.scatter([best_depth], [best_row["CV_R2_均值"]], s=60, color="#b23a48", zorder=5)
ax2.annotate("R² = {:.3f}".format(best_row["CV_R2_均值"]), (best_depth, best_row["CV_R2_均值"]),
             xytext=(10, 14), textcoords="offset points", ha="left", fontsize=9, color="#b23a48")
ax2.set_xlabel("树的最大深度 max_depth")
ax2.set_ylabel("空间 CV 的 R²")
ax2.set_xticks(depths)
ax2.set_title("(b) 同折 R²（误差条为折间 ±1 标准差）")
ax2.grid(axis="y", color=gs.GRID, lw=0.6, alpha=0.85)
ax2.set_axisbelow(True)

fig.suptitle("树越深，训练误差越低；空间验证误差却在某个深度后停滞或回升",
             fontsize=13.5, fontweight="semibold", y=1.02)
gs.save(fig, "06_depth_curve.png")
plt.show()

**怎么读这张图**：灰虚线从深度 1 的 1.305 一路降到深度 12 的 0.592，训练误差几乎单调下降——树越深，越能把训练区的小块差异记住。蓝线（空间折 RMSE 的折均值）却在深度 1 最低（1.373），深度 2 升到 1.520，深度 4–5 回落到 1.419 与 1.411，之后又慢慢升到深度 12 的 1.520。浅色带是折间 ±1 标准差（0.115–0.286），带子很宽，说明 5 个空间折本身就覆盖了不同的地块。

按“折均 RMSE 最小”的规则，最佳深度是 **1**；深度 4–5 与它的差距约 0.04，远小于折间波动，不能当成真实改善。所以结论不是“树越浅越好”，而是：**在这 5 个变量和这块训练区上，超过 1 层之后增加的分裂主要在记训练区的局部差异，没有带来可迁移到邻近区域的空间信息**。训练误差与 CV 误差之间越拉越大的缺口，就是过拟合的可视化。右侧 R² 同理：深度 1 的折均 R² 为 0.238，深度 12 只有 0.088。

为什么深度 2 反而比深度 1 差？看 `06_depth_curve.csv` 的折标准差就知道，这一跳大约是 0.15，与折间波动同量级；5 个空间折的样本量只有 1010–1418，单折的 RMSE 很容易上下摆动 0.1–0.3。把这条曲线读成“深度 1 精确最优”是过度解读，读成“树深超过 1 层后没有稳定收益”才是稳健的结论。

用偏差—方差的语言说：深度 1 的树偏差大（只能把区域分成两片），方差小；加深会降低偏差，却迅速抬高方差——叶子越碎，越依赖少数空间块支撑，换一折就变。空间 CV 误差近似是偏差²、方差与不可约噪声之和，本章的最优点落在偏差仍很大的一侧，说明在这 5 个变量下，方差比偏差更早成为瓶颈。

## 6.4　定稿比较：一棵树和一片森林

随机森林主要靠两种随机性让树之间有所不同：

- 每棵树从训练样本中有放回抽样，所以看到的数据不完全一样；
- 每次分裂只尝试一部分特征，避免所有树都依赖同一个变量。

最后把树的预测平均起来，单棵树容易记住的局部噪声就有机会互相抵消。下面用同一套空间折比较，别先假设森林一定每折都赢。


In [ ]:
dt_name = "决策树（depth={}）".format(best_depth)
rf_name = "随机森林（400 棵）"

dt_cv = gm.cv_scores(lambda: gm.tree_model(max_depth=best_depth), train_df)
rf_cv = gm.cv_scores(lambda: gm.forest_model(n_estimators=400, min_samples_leaf=3), train_df)

# 测试区到此才第一次使用：模型已按训练区 CV 定稿，不再依据测试分数改动
dt_model, dt_test_pred, dt_test = gm.fit_and_test(
    lambda: gm.tree_model(max_depth=best_depth), train_df, test_df)
rf_model, rf_test_pred, rf_test = gm.fit_and_test(
    lambda: gm.forest_model(n_estimators=400, min_samples_leaf=3), train_df, test_df)

fold_scores = pd.concat([
    dt_cv.assign(模型=dt_name),
    rf_cv.assign(模型=rf_name),
], ignore_index=True)
fold_scores = fold_scores.rename(columns={"fold": "折", "n": "像元数"})
fold_scores = fold_scores[["模型", "折", "像元数", "MAE", "RMSE", "R2"]]
fold_scores.to_csv(gd.TABLES / "06_fold_scores.csv", index=False, encoding="utf-8-sig")

def summarize(name, cv, test_metrics):
    return [
        {"模型": name, "评估": "空间 CV（5 折合计）", "像元数": int(cv["n"].sum()),
         "MAE": cv["MAE"].mean(), "RMSE": cv["RMSE"].mean(), "R2": cv["R2"].mean(),
         "MAE_折标准差": cv["MAE"].std(ddof=1), "RMSE_折标准差": cv["RMSE"].std(ddof=1),
         "R2_折标准差": cv["R2"].std(ddof=1)},
        {"模型": name, "评估": "独立测试区", "像元数": len(test_df),
         "MAE": test_metrics["MAE"], "RMSE": test_metrics["RMSE"], "R2": test_metrics["R2"],
         "MAE_折标准差": np.nan, "RMSE_折标准差": np.nan, "R2_折标准差": np.nan},
    ]

metrics_df = pd.DataFrame(summarize(dt_name, dt_cv, dt_test) +
                          summarize(rf_name, rf_cv, rf_test))
metrics_df.to_csv(gd.TABLES / "06_metrics.csv", index=False, encoding="utf-8-sig")
print(metrics_df.round(3).to_string(index=False))

pred_df = test_df[["row", "col", "x", "y", "lst"]].rename(columns={"lst": "观测LST"}).copy()
pred_df["决策树预测"] = dt_test_pred
pred_df["随机森林预测"] = rf_test_pred
pred_df["决策树残差"] = pred_df["决策树预测"] - pred_df["观测LST"]
pred_df["随机森林残差"] = pred_df["随机森林预测"] - pred_df["观测LST"]
pred_df.to_csv(gd.TABLES / "06_test_predictions.csv", index=False, encoding="utf-8-sig")

compare_path = gs.model_comparison(
    {dt_name: dt_cv["RMSE"].to_numpy(), rf_name: rf_cv["RMSE"].to_numpy()},
    {dt_name: dt_test["RMSE"], rf_name: rf_test["RMSE"]},
    out_name="06_model_compare.png",
    title="同一组空间折上的 RMSE：单棵树与随机森林（菱形为独立测试区）",
)
display(Image(filename=str(compare_path), width=780))

**结果解读**：森林的折均 RMSE 是 1.299，低于单树的 1.373；折均 MAE 0.991 对 1.088，折均 R² 0.357 对 0.238。逐折看，森林在第 2、3、4 折明显更好（RMSE 1.249、0.991、1.138 对 1.459、1.311、1.267），在第 1 折反而更差（1.567 对 1.297），第 5 折基本打平（1.547 对 1.530）——所以“森林更稳”并不是每一折都赢。

真正说明问题的是独立测试区：单树的 RMSE 从折均 1.373 涨到 2.019（+47%），R² 从 0.238 掉到 −0.149；森林只从 1.299 涨到 1.471（+13%），R² 仍有 0.390。面对整体偏暖、与训练区不完全同分布的测试区，单树的“两级台阶”式预测失效得更彻底，而 400 棵树平均后的预测退化得少得多。这就是森林“稳”的实际含义：**不是每一折都更低，而是换一个区域时不容易崩**。

也要如实说明一个反例：本例森林的折间 RMSE 标准差（0.253）反而大于单树（0.115）。原因是深度 1 的单树只有两个叶子，预测几乎是两个常数，折间差异被“压平”了——这种低方差是迟钝，不是泛化能力；森林会跟着各折的局部关系变化，折间差异因此显现。判断稳不稳，要同时看折均、逐折胜负和测试区的退化幅度，不能只挑一个指标。

## 6.5 把森林的预测放回地图

表格里的 RMSE 是一个数，地图会告诉我们误差落在哪里。用 `gd.test_window()` 裁出测试区块，用 `gd.grid_from_predictions()` 把 1360 个预测值放回像元位置，再画观测、预测与残差三联图。残差统一为“预测 − 观测”：正值表示模型报高了。

In [ ]:
window = gd.test_window(test_df)
rf_grid = gd.grid_from_predictions(window, rf_test_pred)
resid = rf_grid - window["obs"]

finite = np.isfinite(window["obs"])
print("测试窗口尺寸:", window["shape"], "，有效像元:", int(finite.sum()))
print("观测与预测的像元顺序一致:", np.allclose(window["obs"][finite], test_df["lst"].to_numpy()))
print("残差（预测 − 观测）均值 = {:.3f}，中位数 = {:.3f}".format(
    np.nanmean(resid), np.nanmedian(resid)))

triptych_path = gs.plot_triptych(
    window["obs"], rf_grid, resid, window["extent"],
    title="随机森林在独立测试区：观测、预测与残差（测试 RMSE = {:.2f}）".format(rf_test["RMSE"]),
    out_name="06_triptych_rf.png",
    add_bar=True,
)
display(Image(filename=str(triptych_path), width=980))

**读图**：三幅图共用同一观测/预测色标。观测图里能看到明显的冷暖斑块（左上 47.95°N 附近的一条冷带、47.80°N 附近的暖斑），预测图则被平滑成几大块，极端值被明显压缩——树模型只会输出训练区叶子的均值，不会外推到训练区没出现过的数值。残差图以 0 为中心，本章随机森林的残差均值为 +0.228、中位数 +0.132，整体略偏高，但在冷带附近成片为正（把冷区报暖了），在另一侧成片为负。残差不是随机撒点，说明还有空间结构没被 5 个输入变量解释；这也是第 04 章强调空间折、而不是随机折的原因。

## 6.6 地理视角：树在空间上到底学到了什么

- **树的分裂是“阈值”，不是“梯度”**：它学到的是“气温高于某个值、且 NDVI 低于某个值的地块，平均 LST 更高”，这在地学上很像按地形—植被组合分区，而不是物理方程。
- **空间自相关会让训练误差骗人**：相邻像元高度相似，深树的叶子可以把一小片训练区“整片记住”，但换到测试区就失效。所以本章的所有选择都看空间折，不看随机折，更不看训练误差。
- **森林的稳来自“平均”，不是“更聪明”**：它没有引入新的地理知识，只是降低了单棵树对抽样波动的敏感度；输入变量没覆盖的过程（如灌溉、地表覆盖类型、天气过程）它同样学不到。
- **测试区均值更高，但误差不是简单的“整体偏低”**：训练区目标均值约 11.95，测试区约 12.83；森林在测试区的预测均值约 13.06，残差均值 +0.23。看起来是“报高了”，三联图却显示这是预测被压向训练区中段的结果——冷斑被明显报暖（正残差），暖斑又被报冷（负残差），一平均就抵消了大半。分布差异不能只看均值；树模型对训练区没出现过的极端值尤其保守。这也再次说明本案例是**留出区域估计的方法演示**，不是甘南县 LST 产品。

## 6.7 常见坑

1. **用训练误差挑深度**：训练 RMSE 几乎随深度单调下降，照着它选一定会选到最深的树。
2. **用测试区反复试深度**：测试区一旦参与调参，就不再是独立评估；本章只在最后评估一次。
3. **把 `max_depth` 当成唯一正则化**：`min_samples_leaf`、`max_features` 同样控制复杂度；本章固定它们，只观察深度的作用。
4. **忽略缺测**：训练区高程缺 38、气温缺 91 个像元（全目标区分别为 41 与 97），管道里的中位数填补只在训练折内拟合；不要用全体数据先填补再切折。
5. **把特征重要性当因果**：树的分裂次数受变量尺度和相关性影响，重要性高只说明“好用”，不说明“导致”。
6. **忘记单位**：本章所有数值都是原始像元值，元数据没有给出单位，不能写成 °C。

## 6.8　练习

打开 `outputs/tables/06_fold_scores.csv` 和 `06_metrics.csv`，先逐折比较决策树与随机森林的 RMSE。森林在哪些折上改善最多？有没有哪一折反而变差？

再把同一深度的树换成随机 5 折，和空间 5 折并排看。想一想：相邻像元被分到两边以后，验证任务发生了什么变化？这次练习只用训练区，不需要重新评估测试区。


## 小结

- 回归树用方差下降选分裂，叶子预测均值；深度越大，训练误差越低，但空间外推误差会在某个深度后停滞甚至回升。
- 本章按训练区 5 个空间折的 CV RMSE 选出最佳深度，测试区只在定稿后评估一次。
- 随机森林用 bagging 与特征随机性制造“有差异的树”，再平均：折均误差更低，换到独立测试区时 RMSE 只上升 13%，而单树上升 47%。
- 测试区整体偏暖（均值约 12.83 vs 训练区 11.95），残差与误差里含有分布差异，不能全部归因于模型。
- 树模型给出的仍是相关关系与分区规则，不是地表温度形成机制的因果解释。

### 运行记录

下面这一格把本章的核、Python、耗时、最佳深度、关键指标、图件与表格清单写进 `outputs/tables/06_run_summary.json`，方便复核；数字全部来自本次实际执行。

In [ ]:
import json

elapsed = time.perf_counter() - T0

def brief(cv, test_metrics):
    return {
        "CV_RMSE_折均": round(float(cv["RMSE"].mean()), 3),
        "CV_RMSE_折标准差": round(float(cv["RMSE"].std(ddof=1)), 3),
        "CV_MAE_折均": round(float(cv["MAE"].mean()), 3),
        "CV_R2_折均": round(float(cv["R2"].mean()), 3),
        "测试区_RMSE": round(float(test_metrics["RMSE"]), 3),
        "测试区_MAE": round(float(test_metrics["MAE"]), 3),
        "测试区_R2": round(float(test_metrics["R2"]), 3),
    }

summary = {
    "notebook": "notebooks/06_决策树与随机森林.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "run_time_local": pd.Timestamp.now(tz="Asia/Shanghai").isoformat(timespec="seconds"),
    "notebook_elapsed_seconds": round(elapsed, 1),
    "data": "data/processed/training_table_splits.csv（固定空间折与测试区）",
    "split": {
        "训练区像元": int(len(train_df)),
        "测试区像元": int(len(test_df)),
        "训练区目标均值": round(float(train_df["lst"].mean()), 3),
        "测试区目标均值": round(float(test_df["lst"].mean()), 3),
    },
    "selection_rule": "只按训练区 5 个空间折的 CV RMSE 折均值选择树深；测试区在模型定稿后只评估一次",
    "best_depth": int(best_depth),
    "key_metrics": {
        dt_name: brief(dt_cv, dt_test),
        rf_name: brief(rf_cv, rf_test),
    },
    "figures": [
        "outputs/figures/06_shallow_tree.png",
        "outputs/figures/06_depth_curve.png",
        "outputs/figures/06_model_compare.png",
        "outputs/figures/06_triptych_rf.png",
    ],
    "tables": [
        "outputs/tables/06_depth_curve.csv",
        "outputs/tables/06_fold_scores.csv",
        "outputs/tables/06_metrics.csv",
        "outputs/tables/06_test_predictions.csv",
        "outputs/tables/06_run_summary.json",
    ],
    "unverified": [
        "LST 与 5 个变量的单位未在元数据中注明，全文按“原始像元值”表述",
        "LST 来自 2021 年 LST_Day，具体月日未知；TEM 为 2021 年，时间聚合方式未知",
        "降水与土壤水分为 2020 年，与目标不同期；NDVI 无时间血缘",
        "测试区是固定中南区块（1360 像元）的单次留出评估，结论只适用于本案例",
        "未核实地表覆盖、灌溉、天气过程等未进入模型的影响",
    ],
}
gd.write_json(summary, "06_run_summary.json")
print(json.dumps(summary, ensure_ascii=False, indent=2))